In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: PbSO4, XRD

This example demonstrates a more advanced use of the EasyDiffraction
library by explicitly creating and configuring structures and
experiments before adding them to a project. It could be more suitable
for users who are interested in creating custom workflows. This
tutorial provides minimal explanation and is intended for users
already familiar with EasyDiffraction.

The tutorial covers a Rietveld refinement of PbSO4 crystal structure
based on laboratory X-ray powder diffraction data.

## 🛠️ Import Library

In [ ]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [ ]:
struct = StructureFactory.from_scratch(name='pbso4')

### Set Space Group

In [ ]:
struct.space_group.name_h_m = 'P n m a'

### Set Unit Cell

In [ ]:
struct.cell.length_a = 8.48
struct.cell.length_b = 5.40
struct.cell.length_c = 6.96

### Set Atom Sites

In [ ]:
struct.atom_sites.create(
    id='Pb',
    type_symbol='Pb',
    fract_x=0.1876,
    fract_y=0.25,
    fract_z=0.167,
    adp_type='Biso',
    adp_iso=1.37,
)
struct.atom_sites.create(
    id='S',
    type_symbol='S',
    fract_x=0.0654,
    fract_y=0.25,
    fract_z=0.684,
    adp_type='Biso',
    adp_iso=0.3796,
)
struct.atom_sites.create(
    id='O1',
    type_symbol='O',
    fract_x=0.9082,
    fract_y=0.25,
    fract_z=0.5954,
    adp_type='Biso',
    adp_iso=1.9840,
)
struct.atom_sites.create(
    id='O2',
    type_symbol='O',
    fract_x=0.1935,
    fract_y=0.25,
    fract_z=0.5432,
    adp_type='Biso',
    adp_iso=1.4383,
)
struct.atom_sites.create(
    id='O3',
    type_symbol='O',
    fract_x=0.0811,
    fract_y=0.0272,
    fract_z=0.8086,
    adp_type='Biso',
    adp_iso=1.2808,
)

## 🔬 Define Experiments

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

### Experiment: xrd

#### Download Data

In [ ]:
data_path = download_data('meas-pbso4-xray', destination='data')

#### Create Experiment

In [ ]:
expt = ExperimentFactory.from_data_path(
    name='xrd',
    data_path=data_path,
    radiation_probe='xray',
)

#### Set Instrument

In [ ]:
expt.instrument.setup_wavelength = 1.540560
expt.instrument.setup_wavelength_2 = 1.544400
expt.instrument.setup_wavelength_2_to_1_ratio = 0.5

expt.instrument.setup_polarization_coefficient = 0.58
expt.instrument.setup_monochromator_twotheta = 28

expt.instrument.calib_twotheta_offset = -0.02

#### Set Peak Profile

In [ ]:
expt.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'

In [ ]:
expt.peak.broad_gauss_u = 0.03
expt.peak.broad_gauss_v = -0.04
expt.peak.broad_gauss_w = 0.01
expt.peak.broad_lorentz_y = 0.06

expt.peak.asym_beba_a0 = -0.23
expt.peak.asym_beba_b0 = -0.03

expt.peak.cutoff_fwhm = 6

#### Set Excluded Regions

In [ ]:
expt.excluded_regions.create(id='1', start=0, end=15)
expt.excluded_regions.create(id='2', start=160, end=180)

#### Set Background

Select background type.

In [ ]:
expt.background.type = 'chebyshev'

Add Chebyshev background terms.

In [ ]:
for id, x, y in [
    ('1', 0, 149.0),
    ('2', 1, 67.0),
    ('3', 2, 9.0),
    ('4', 3, 10.0),
    ('5', 4, -5.0),
    ('6', 5, -9.0),
]:
    expt.background.create(id=id, order=x, coef=y)

#### Set Linked Structures

In [ ]:
expt.linked_structures.create(structure_id='pbso4', scale=0.001)

## 📦 Define Project

The project object is used to manage structures, experiments, and
analysis.

### Create Project

In [ ]:
project = Project(name='pbso4_xray')

### Add Structure

In [ ]:
project.structures.add(struct)

### Add Experiment

In [ ]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section outlines the analysis process, including how to configure
calculation and fitting engines.

### Set Free Parameters

Set structure parameters to be optimized.

In [ ]:
struct.cell.length_a.free = True
struct.cell.length_b.free = True
struct.cell.length_c.free = True

Set experiment parameters to be optimized.

In [ ]:
expt.linked_structures['pbso4'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

expt.peak.asym_beba_a0.free = True
expt.peak.asym_beba_b0.free = True

for term in expt.background:
    term.coef.free = True

### Run Fitting

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

#### Display Correlations

In [ ]:
project.display.fit.correlations()

### Display Pattern

In [ ]:
project.display.pattern(expt_name='xrd')

In [ ]:
project.display.pattern(expt_name='xrd', x_min=77.6, x_max=82.2)

## 💾 Save Project

In [ ]:
project.save_as(dir_path='projects/refine-pbso4-xray')